In [1]:
import duckdb
import polars as pl
import numpy as np
import pendulum
from pathlib import Path

data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)

print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)
df = df.with_columns(
    artist_name = pl.col("artist_name").str.to_lowercase(),
    track_name = pl.col("track_name").str.to_lowercase(),
    album_name = pl.col("album_name").str.to_lowercase()
)
df.head()

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc
str,str,str,str,str,str,i64,datetime[μs]
"""elias rønnenfelt""","""""","""close (the dare remix)""","""""","""close - the dare remix""","""""",1742283786,2025-03-18 07:43:06
"""macabre plaza""","""""","""out with the old in with the t…","""""","""till the wheels fall off""","""""",1709815426,2024-03-07 12:43:46
"""desire""","""""","""desire""","""8af35ae0-96af-453c-8e6e-3047ca…","""under your spell""","""17579de2-f1e4-4e0e-a4c7-e18106…",1737304566,2025-01-19 16:36:06
"""tracey brakes""","""4e40592a-3581-42f1-b6e8-ced9ac…","""wand of tens""","""""","""wand of tens""","""""",1677664915,2023-03-01 10:01:55
"""alex g""","""7df4c4de-3181-4a69-bd36-9d6639…","""trick""","""141c833b-c3c4-4e12-8157-bf5435…","""sarah""","""70ad2ffc-5980-4f44-b43b-a8edc3…",1644394331,2022-02-09 08:12:11


## Seasonal patterns — do you listen to certain artists more in winter vs. summer?


In [2]:
df_ = df.filter(pl.col("track_played_utc").dt.year() == 2025)

In [3]:
df_pivot = (
    df_
    .group_by(
        pl.col("artist_name"), 
        pl.col("track_played_utc").dt.strftime("%B").alias("month"),
        pl.col("track_played_utc").dt.month().alias("month_idx")
    )
    .agg(
        scrobbles = pl.len()
    )
    .sort(pl.col("month_idx"), descending=False)
    .pivot(
        on="month",
        index="artist_name",
        values="scrobbles",
        aggregate_function="sum"
    )
)
df_pivot

artist_name,January,February,March,April,May,June,July,August,September,October,November,December
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""infime""",1,1,1,0,2,0,1,0,0,0,0,0
"""night tapes""",2,1,0,0,0,0,0,0,0,0,0,1
"""eera""",17,11,25,7,5,3,2,0,0,0,1,0
"""beth gibbons""",2,1,1,0,0,0,0,2,1,0,0,0
"""wet leg""",3,1,9,0,0,0,1,1,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…
"""rome rome""",0,0,0,0,0,0,0,0,0,0,0,1
"""lil ugly mane""",0,0,0,0,0,0,0,0,0,0,0,9
"""adamn killa""",0,0,0,0,0,0,0,0,0,0,0,1


In [4]:
n = 20
df_top_n_artists = (
    df_
    .group_by(
        pl.col("artist_name"), 
        # pl.col("track_played_utc").dt.strftime("%B").alias("month"),
        # pl.col("track_played_utc").dt.month().alias("month_idx")
    )
    .agg(
        scrobbles = pl.len()
    )
    .sort(pl.col("scrobbles"), descending=True)
).head(20)

df_pivot.join(df_top_n_artists, on="artist_name").sort("scrobbles", descending=True)

artist_name,January,February,March,April,May,June,July,August,September,October,November,December,scrobbles
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""bladee""",26,53,31,23,68,25,38,79,50,42,11,13,459
"""dean blunt""",0,7,86,10,27,46,7,82,58,66,8,14,411
"""bassvictim""",5,90,35,35,12,5,20,0,3,65,73,26,369
"""astrid sonne""",4,2,36,11,159,28,8,11,21,9,8,4,301
"""vegyn""",5,6,2,153,23,68,4,12,3,10,11,4,301
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""mj lenderman""",0,6,129,9,5,6,0,5,4,5,1,4,174
"""jane remover""",36,22,17,73,9,0,3,1,1,0,3,2,167
"""blood orange""",0,0,0,0,0,27,41,50,31,4,3,10,166


In [5]:
(
    df_
    .group_by(
        pl.col("artist_name"), 
        pl.col("track_played_utc").dt.strftime("%B").alias("month"),
        pl.col("track_played_utc").dt.month().alias("month_idx")
    )
    .agg(
        scrobbles = pl.len()
    )
)

artist_name,month,month_idx,scrobbles
str,str,i8,u32
"""matt maltese""","""March""",3,1
"""yehezkel raz""","""February""",2,1
"""misha panfilov""","""September""",9,6
"""oneohtrix point never""","""December""",12,1
"""tevvez""","""June""",6,1
…,…,…,…
"""huerco s.""","""June""",6,1
"""axlr""","""February""",2,1
"""purient""","""August""",8,1


## "Throwback" plays — replaying old favorites after a long gap

In [6]:
# # Cumulative listening per track, so each record gets a number representing the n'th time of listening to a track.
# bv = (
#     df.sort(pl.col("date_played_unix"), descending=False)
#     .filter(pl.col("track_played_utc").dt.year() == year)
#     .filter(pl.col("artist_name") == "bassvictim")
#     .filter(pl.col("track_name").len().over("track_name") > 10)
#     .with_columns(
#         track_nth_scrobble = pl.col("track_name").cum_count().over(pl.col("track_name"))
#     )
# )
# bv.head()

In [7]:
(
    df
    .sort(pl.col("track_played_utc"), descending=False)
    .with_columns(
        last_played_delta = (
            pl.col("track_played_utc") - pl.col("track_played_utc").shift(1).over("track_name")
            ).dt.total_days(fractional=True).fill_null(0).ceil()
    )
).filter(pl.col("artist_name") == "bladee")

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,last_played_delta
str,str,str,str,str,str,i64,datetime[μs],f64
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""drama (feat. charli xcx)""","""""","""drama (feat. charli xcx)""","""""",1621494114,2021-05-20 07:01:54,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""gluee""","""250bafde-fd0c-4490-a728-5f741f…","""ebay""","""""",1621494495,2021-05-20 07:08:15,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""working on dying""","""587b563c-73ff-4d81-b669-b49a2e…","""lordship""","""22de2b38-a027-49fe-8e99-c30809…",1621494644,2021-05-20 07:10:44,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""red light""","""725f97cb-5e7a-4bbb-be43-c65004…","""decay""","""404d136c-174c-4cb5-9fb0-1e8859…",1621494772,2021-05-20 07:12:52,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""working on dying""","""587b563c-73ff-4d81-b669-b49a2e…","""lordship""","""22de2b38-a027-49fe-8e99-c30809…",1621498932,2021-05-20 08:22:12,1.0
…,…,…,…,…,…,…,…,…
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""love is a state""","""3540f010-4c99-4c9e-80e1-3dba8f…","""eyelash""","""6253d641-5088-4908-96df-100b4e…",1774283687,2026-03-23 16:34:47,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""eversince""","""96e5ccf7-2ba9-4f94-944d-79ca01…","""rip""","""ecd0d043-fa8e-44e1-926b-94e108…",1774283914,2026-03-23 16:38:34,517.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""sesame street""","""2c0a269b-535a-47fe-97b2-fa4d65…","""sesame street""","""1907b9ad-2964-4889-9df3-be44ae…",1774355995,2026-03-24 12:39:55,26.0


In [14]:
(
    df
    .filter(pl.col("track_played_utc").dt.year() == 2025)
    .group_by(pl.col("track_name"), pl.col("artist_name"))
    .agg(
        n_scrobbles = pl.len(),
        first_played = pl.col("track_played_utc").min(),
        last_played = pl.col("track_played_utc").max(),
    )
    .sort(pl.col("n_scrobbles"), descending=True)
)

track_name,artist_name,n_scrobbles,first_played,last_played
str,str,u32,datetime[μs],datetime[μs]
"""5""","""dean blunt""",59,2025-02-14 10:02:11,2025-12-17 16:14:16
"""flash""","""2hollis""",47,2025-04-04 12:02:50,2025-12-01 14:47:24
"""react [r tyler edit]""","""dj_dave""",46,2025-01-23 16:16:02,2025-12-15 17:26:52
"""bluey vuitton""","""babyfather""",44,2025-09-28 09:15:03,2025-11-24 15:51:20
"""woman lake""","""snuggle""",43,2025-06-17 07:23:51,2025-12-18 07:36:53
…,…,…,…,…
"""starjumps""","""earthmelon""",1,2025-08-10 13:06:07,2025-08-10 13:06:07
"""noblest strive""","""bladee""",1,2025-01-08 15:53:19,2025-01-08 15:53:19
"""(=o_o=)""","""polygonia""",1,2025-05-10 10:56:12,2025-05-10 10:56:12


In [16]:
df_artists = (
    df
    .filter(pl.col("track_played_utc").dt.year() == 2025)
    .group_by(pl.col("artist_name"), pl.col("track_name"))
    .agg(
      track_scrobbles = pl.len(), # scrobbles per track
    )
    .with_columns(
      artist_scrobbles=pl.col("track_scrobbles").sum().over("artist_name") # total artist scrobbles
    )
    .sort(pl.col("artist_scrobbles"), descending=True)
)
top_n_artists = df_artists.select("artist_name").unique(maintain_order=True).head(25)
df_top_n_artists = ( 
    df_artists
    .join(top_n_artists, on="artist_name", how="semi") 
    .sort("track_scrobbles", descending=True)
)